# Kernel directive

A kernel directive is single line comment character sequence (`##@`) at the top of a block of code. 

``` python
##@<execute mode> <option1_name=value>, <option2_name=value>....
# Only valid for this code block
```

The kernel directive only applies to the code block in which it is written and can modify how and where (namespace) the code is executed.
 
The first part of the directive is the execute mode, which is one of `task | thread | queue`, `queue` is default if the execute mode is omitted. Following the execute mode are options relevant to the execute mode. The `namespace_id` option is relevant to all execute modes.

- `##@task namespace_id=<value>`
- `##@thread thread_name=<value> namespace_id=<value>`
- `##@queue namespace_id=<value>` or `##@ namespace_id=<value>`


Provided the frontend supports it (Jupyterlab does, VS code doesn't), multiple cells can be run concurrently.

## Example

**This example requires ipywidgets!**

Lets define a function that we'll reuse for the remainder of the notebook.

In [ ]:
async def demo():
    import threading

    import anyio
    from ipywidgets import Button

    from async_kernel import Caller

    print(f"Thread name: '{threading.current_thread().name}'")
    button = Button(description="Finish")
    event = anyio.Event()
    caller = Caller()  # Use caller so the #@thread example works.
    # This is because widget messages are received by the shell in the main  thread, but the anyio event is being waited in this thread.
    button.on_click(lambda _: caller.call_no_context(event.set))
    display(button)
    await event.wait()
    button.close()
    print(f"Finished ... thread name: '{threading.current_thread().name}'")
    return "Finished"

Lets run it normally (queue)

In [ ]:
await demo()

Calling a code block without a kernel directive is equivalent to the kernel directive `##@queue namespace=`.

### Execute mode: task
``` python
#@task
...
```

The `task` mode instructs the kernel to execute the code in a task separate to the queue, Both `task` and `thread` execute modes can be started when the kernel is *busy executing*. There is no imposed limitation on the number of tasks (or threads) that can be run concurrently. 

In [ ]:
##@task
await demo()

### Execute mode: thread
``` python
#@thread, <thread_name=name>
...
```


In [ ]:
##@thread thread_name=my thread
await demo()

In [ ]:
##@thread thread_name=my thread
%threads # async kernel provides the `threads` magic

### option: namespace_id

Multiple namespaces are supported by async kernel. Pass `namspace_id=<value>` in the header directive to use a different namespace. The globals and locals namespace is the same dict for async kernel. All namespace dicts are stored at `shell.namespaces` should you need to access the objects from another namespace.

In [ ]:
##@task namespace_id=my namespace
from async_kernel import Kernel

kernel = Kernel()

a = 10

assert a in kernel.shell.namespaces["my namespace"].values()
assert a not in kernel.shell.namespaces[""].values()

In [ ]:
##@task
from async_kernel import Kernel

kernel = Kernel()

list(kernel.shell.namespaces)

In [ ]:
##@task namespace_id=my namespace
a * 2